# Factor Backtest Walkthrough — P/E

Step-by-step decomposition of `run_backtest('pe', ...)`.  
Each intermediate DataFrame is surfaced so you can inspect what the engine actually does.

**Flow:**
1. Load raw data once
2. Build rebalance schedule
3. PIT alignment — which filing is 'known' at date T
4. Cross-section at T — compute P/E for all tickers
5. Quintile assignment
6. Entry / exit prices and forward returns
7. IC at T — Spearman rank correlation
8. Full loop over all rebalance dates
9. IC series chart
10. Quintile cumulative return chart

In [1]:
import datetime

import numpy as np
import pandas as pd
from scipy import stats

from irp.factors._cols import (
    TICKER, REPORT_DATE, PUBLISH_DATE,
    PRICE_CLOSE, PRICE_DATE, PRICE_TICKER,
)
from irp.factors._pit import pit_latest, pit_prepare, pit_price
from irp.factors.backtest import _price_at
from irp.factors.valuation import compute_valuation
from irp.query.simfin import fundamentals
from irp.query.yahoo import prices as yahoo_prices

VARIANT = 'A'
START   = datetime.date(2018, 12, 31)
END     = datetime.date(2023, 12, 31)
HORIZON = 252   # calendar days (~1 year forward return)
FREQ    = 'QE'  # quarterly rebalance

## Cell 1 — Load raw data once

All DB queries happen here. The rest of the notebook operates on these in-memory DataFrames.

`pit_prepare` parses dates and adds an `_eff` (effective public date) column so that
subsequent PIT lookups in the loop skip repeated parsing.

In [2]:
raw_income   = fundamentals(None, 'income',   VARIANT)
raw_balance  = fundamentals(None, 'balance',  VARIANT)
raw_cashflow = fundamentals(None, 'cashflow', VARIANT)
raw_prices   = yahoo_prices(None)

raw_income_p   = pit_prepare(raw_income,   'fundamental')
raw_balance_p  = pit_prepare(raw_balance,  'fundamental')
raw_cashflow_p = pit_prepare(raw_cashflow, 'fundamental')
raw_prices_p   = pit_prepare(raw_prices,   'price')

print(f"Income    rows: {len(raw_income):>10,}  |  tickers: {raw_income[TICKER].nunique():,}")
print(f"Balance   rows: {len(raw_balance):>10,}  |  tickers: {raw_balance[TICKER].nunique():,}")
print(f"Cashflow  rows: {len(raw_cashflow):>10,}  |  tickers: {raw_cashflow[TICKER].nunique():,}")
print(f"Prices    rows: {len(raw_prices):>10,}  |  tickers: {raw_prices[PRICE_TICKER].nunique():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Income    rows:     58,318  |  tickers: 5,053
Balance   rows:     58,314  |  tickers: 5,053
Cashflow  rows:     58,318  |  tickers: 5,053
Prices    rows: 37,823,943  |  tickers: 12,355


## Cell 2 — Rebalance schedule

Quarterly end-of-quarter dates between START and END.  
At each of these dates we will form a portfolio.

In [6]:
rebalance_dates = [
    ts.date()
    for ts in pd.date_range(START, END, freq=FREQ)
]
print(f"{len(rebalance_dates)} rebalance dates")
for d in rebalance_dates:
    print(f"  {d}")

21 rebalance dates
  2018-12-31
  2019-03-31
  2019-06-30
  2019-09-30
  2019-12-31
  2020-03-31
  2020-06-30
  2020-09-30
  2020-12-31
  2021-03-31
  2021-06-30
  2021-09-30
  2021-12-31
  2022-03-31
  2022-06-30
  2022-09-30
  2022-12-31
  2023-03-31
  2023-06-30
  2023-09-30
  2023-12-31


## Cell 3 — PIT alignment demo

**Point-in-time (PIT)** means: at date T, we can only use filings that were
**publicly available** on or before T.

The effective public date (`_eff`) is:
- `Publish Date` if SimFin recorded it
- `Report Date + 60 days` as a conservative fallback

Using `Report Date` alone would introduce **lookahead bias** — we would assume
a company's annual results were known on the day the fiscal year ended,
but in reality the 10-K filing arrives weeks later.

`pit_latest` selects, **per ticker**, the most-recent filing whose `_eff <= T`.

In [13]:
T = rebalance_dates[0]
print(f"Snapshot date T = {T}")
print()

aapl_inc = raw_income_p[raw_income_p[TICKER] == 'AAPL'].copy()
print("All AAPL income filings (with effective public date):")
display(
    aapl_inc[[TICKER, REPORT_DATE, PUBLISH_DATE, '_eff', 'Revenue']]
    .tail(8)
    .reset_index(drop=True)
)

inc_T = pit_latest(raw_income_p,   T)
bal_T = pit_latest(raw_balance_p,  T)
cf_T  = pit_latest(raw_cashflow_p, T)
px_T  = pit_price(raw_prices_p,    T)

print(f"Tickers with PIT-safe filing at {T}:")
print(f"  Income:    {len(inc_T):,}")
print(f"  Balance:   {len(bal_T):,}")
print(f"  Cashflow:  {len(cf_T):,}")
print(f"  Prices:    {len(px_T):,}")
print()

aapl_selected = inc_T[inc_T[TICKER] == 'AAPL'][[TICKER, REPORT_DATE, PUBLISH_DATE, 'Revenue']]
print(f"AAPL filing selected by pit_latest at {T}:")
display(aapl_selected)

Snapshot date T = 2018-12-31

All AAPL income filings (with effective public date):


,Ticker,Report Date,Publish Date,_eff,Revenue
0,AAPL,2018-09-30,2018-11-05,2018-11-05,265595000000
1,AAPL,2019-09-30,2019-10-31,2019-10-31,260174000000
2,AAPL,2020-09-30,2020-10-30,2020-10-30,274515000000
3,AAPL,2021-09-30,2021-10-29,2021-10-29,365817000000
4,AAPL,2022-09-30,2022-10-28,2022-10-28,394328000000
5,AAPL,2023-09-30,2023-11-03,2023-11-03,383285000000
6,AAPL,2024-09-30,2024-11-01,2024-11-01,391035000000
7,AAPL,2025-09-30,2025-10-31,2025-10-31,416161000000


Tickers with PIT-safe filing at 2018-12-31:
  Income:    3,369
  Balance:   3,367
  Cashflow:  3,373
  Prices:    5,990

AAPL filing selected by pit_latest at 2018-12-31:


,Ticker,Report Date,Publish Date,Revenue
12,AAPL,2018-09-30,2018-11-05,265595000000


## Cell 4 — Cross-section at T: compute P/E

`compute_valuation` joins income + balance + cashflow + prices (all PIT-safe)
and derives the valuation ratios.  Returns one row per ticker.

P/E = market cap / net income. Negative P/E (loss-making companies) is economically
meaningless for a value sort, so we filter it out.

In [20]:
xs_T = compute_valuation(inc_T, bal_T, cf_T, px_T)[['pe', 'mktcap']]

pe_T = xs_T['pe'].dropna()
pe_T = pe_T[pe_T > 0]

print(f"Tickers with valid (positive) P/E at {T}: {len(pe_T):,}")
print(f"P/E range: {pe_T.min():.2f} – {pe_T.max():.1f}   median: {pe_T.median():.1f}")
print()

pe_T_df = xs_T.loc[pe_T.index].copy()

print("10 cheapest stocks (lowest P/E):")
display(pe_T_df.nsmallest(10, 'pe'))

print("10 most expensive stocks (highest P/E):")
display(pe_T_df.nlargest(10, 'pe'))

Tickers with valid (positive) P/E at 2018-12-31: 1,296
P/E range: 0.00 – 63387.1   median: 18.6

10 cheapest stocks (lowest P/E):


,pe,mktcap
Ticker,,
PANL,0.000012,97341199.881058
HIMX,0.013849,432424694.191933
VHI,0.226436,46985393.440836
CIK,0.312669,166355353.818776
CAPR,0.451716,1098312.074453
TEN,0.550528,109004572.622942
STGW,0.554119,134012550.88361
NLY,0.654086,1026656541.432983
VATE,0.695511,-32619475.1923


10 most expensive stocks (highest P/E):


,pe,mktcap
Ticker,,
DAVA,63387.092635,1202770082.751226
SKY,25009.354098,125046770.488066
RCMT,18807.205889,37802483.837056
APLE,12047.891444,2198643805393.218994
RDNT,9095.802645,482077540.18648
VICR,9036.335967,1509068106.559753
WWW,8713.813057,2614143917.08374
SPSC,8146.935797,2859574464.660645
IT,3500.687182,11478753271.179199


## Cell 5 — Quintile assignment at T

The universe is split into 5 equally-sized buckets ranked by P/E.
- **Q1** = cheapest 20% (low P/E)
- **Q5** = most expensive 20% (high P/E)

If value works, Q1 should outperform Q5 over the forward horizon.

In [23]:
pe_T_df = pe_T_df.copy()
pe_T_df['quintile'] = pd.qcut(pe_T_df['pe'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

print("Quintile summary (P/E ranges and counts):")
display(
    pe_T_df.groupby('quintile', observed=True)['pe']
    .agg(count='count', min='min', mean='mean', max='max')
    .round(1)
)

print("\nSample Q1 — cheapest stocks:")
display(pe_T_df[pe_T_df['quintile'] == 'Q1'].head(10))

print("\nSample Q5 — most expensive stocks:")
display(pe_T_df[pe_T_df['quintile'] == 'Q5'].head(10))

Quintile summary (P/E ranges and counts):


,count,min,mean,max
quintile,,,,
Q1,260,0.0,5.5,8.9
Q2,259,8.9,12.1,15.2
Q3,259,15.3,18.6,22.3
Q4,259,22.4,29.5,39.9
Q5,259,40.2,752.7,63387.1



Sample Q1 — cheapest stocks:


,pe,mktcap,quintile
Ticker,,,
AAOI,4.202058,310746396.295967,Q1
ACCO,3.96772,522548731.231689,Q1
ACLS,4.687819,595160774.490356,Q1
ACRE,5.083349,154569385.163779,Q1
ACTG,6.810739,151062196.726873,Q1
AGM,5.995136,506481045.474243,Q1
AGNC,3.215758,2479349173.879623,Q1
AGX,6.824238,491420191.268921,Q1
AIV,7.174577,2265544851.171227,Q1



Sample Q5 — most expensive stocks:


,pe,mktcap,quintile
Ticker,,,
A,65.800991,20793113040.924072,Q5
AAT,46.061719,1848548887.683868,Q5
ABT,232.01571,110671493762.969971,Q5
ACIW,643.625217,3305015489.112854,Q5
ADBE,43.756878,113374070272.750854,Q5
ADUS,66.005957,788969208.076477,Q5
AEHR,60.838294,32122619.239569,Q5
AIR,81.720205,1274835194.396973,Q5
AIRG,89.992292,102681204.84898,Q5


## Cell 6 — Entry and exit prices for T

**Entry price**: closing price on (or before) T  
**Exit price**: closing price on (or before) T + HORIZON days

Forward return = log(exit / entry). Log returns are additive over time and
approximately normally distributed — preferred over simple returns for analysis.

In [25]:
T_entry = pd.Timestamp(T)
T_exit  = T_entry + pd.Timedelta(days=HORIZON)

p0 = _price_at(raw_prices_p, T_entry)   # DataFrame: index=Ticker, cols=[Close, Date]
p1 = _price_at(raw_prices_p, T_exit)

p0 = p0.rename(columns={PRICE_CLOSE: 'entry_price', PRICE_DATE: 'entry_date'})
p1 = p1.rename(columns={PRICE_CLOSE: 'exit_price',  PRICE_DATE: 'exit_date'})
prices_T = p0.join(p1, how='inner')
prices_T['fwd_ret'] = np.log(prices_T['exit_price'] / prices_T['entry_price'])

print(f"Entry date: {T}  →  Exit window: up to {T_exit.date()}")
print(f"Tickers with both entry and exit price: {len(prices_T):,}")
print(f"\nForward return distribution:")
print(prices_T['fwd_ret'].describe().round(4))
print()
display(prices_T.head(10))

Entry date: 2018-12-31  →  Exit window: up to 2019-09-09
Tickers with both entry and exit price: 5,990

Forward return distribution:
count    5990.0000
mean        0.0607
std         0.4518
min        -4.7302
25%        -0.0012
50%         0.1128
75%         0.2117
max         3.0073
Name: fwd_ret, dtype: float64



,entry_price,entry_date,exit_price,exit_date,fwd_ret
Ticker,,,,,
A,63.978809,2018-12-31,70.651031,2019-09-09,0.099201
AA,25.347107,2018-12-31,19.358393,2019-09-09,-0.269539
AAAU,12.820000,2018-12-31,14.970000,2019-09-09,0.155042
AACG,0.920000,2018-12-31,1.860000,2019-09-09,0.703958
AADR,36.931423,2018-12-31,45.600693,2019-09-09,0.210860
AAL,31.599043,2018-12-31,28.243738,2019-09-09,-0.112255
AAME,2.283186,2018-12-31,2.626137,2019-09-09,0.139942
AAMI,10.019523,2018-12-31,9.679688,2019-09-09,-0.034506
AAOI,15.430000,2018-12-31,10.580000,2019-09-09,-0.377348


## Cell 7 — IC at T: Spearman rank correlation

**Information Coefficient (IC)** = Spearman rank correlation between the factor
value and the realized forward return.

- IC = −1 means the factor perfectly predicts returns (in reverse rank order)
- IC = 0 means no predictive power
- IC = +1 means perfect positive rank correlation

For P/E, we expect **negative IC**: low P/E (cheap) → higher returns.
A mean IC of −0.03 to −0.05 is considered a useful signal in practice.

In [34]:
merged_T = (
    pe_T_df[['pe']]
    .join(prices_T[['fwd_ret']], how='inner')
    .dropna()
)

ic_val, pval = stats.spearmanr(merged_T['pe'], merged_T['fwd_ret'])
print(f"Observations: {len(merged_T):,}")
print(f"Spearman IC  = {ic_val:+.4f}")
print(f"p-value      = {pval:.4f}")
print(f"Significant? {'yes' if pval < 0.05 else 'no'} (p < 0.05)")
print()
print("Interpretation: IC < 0 means cheap stocks (low PE) tended to have higher returns.")
print()

merged_T = merged_T.copy()
merged_T['quintile'] = pd.qcut(
    merged_T['pe'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
)
print(f"Mean forward return by quintile at {T}:")
display(
    merged_T.groupby('quintile', observed=True)['fwd_ret']
    .agg(count='count', mean='mean', std='std')
    .round(4)
)

Observations: 1,296
Spearman IC  = +0.0769
p-value      = 0.0056
Significant? yes (p < 0.05)

Interpretation: IC < 0 means cheap stocks (low PE) tended to have higher returns.

Mean forward return by quintile at 2018-12-31:


,count,mean,std
quintile,,,
Q1,260,0.0858,0.3183
Q2,259,0.1396,0.2292
Q3,259,0.1566,0.2287
Q4,259,0.1553,0.2407
Q5,259,0.1448,0.2950


## Cell 8 — Full loop over all rebalance dates

Cells 3–7 showed the logic at a single date T.  
Here we repeat it for every date in the rebalance schedule.

This is exactly what `run_backtest()` does internally, minus caching.

In [ ]:
ic_records   = []
qret_records = []

for T in rebalance_dates:
    # --- PIT-safe fundamentals and price at T ---
    inc = pit_latest(raw_income_p,   T)
    bal = pit_latest(raw_balance_p,  T)
    cf  = pit_latest(raw_cashflow_p, T)
    px  = pit_price(raw_prices_p,    T)
    if any(x.empty for x in [inc, bal, cf, px]):
        continue

    # --- P/E cross-section ---
    xs = compute_valuation(inc, bal, cf, px)
    pe = xs['pe'].dropna()
    pe = pe[pe > 0]
    if len(pe) < 20:
        continue

    # --- Forward returns ---
    T_ts = pd.Timestamp(T)
    p0   = _price_at(raw_prices_p, T_ts)
    p1   = _price_at(raw_prices_p, T_ts + pd.Timedelta(days=HORIZON))
    if p0.empty or p1.empty:
        continue

    fwd = {}
    for tk in pe.index.intersection(p0.index).intersection(p1.index):
        ep = float(p0.loc[tk, PRICE_CLOSE])
        xp = float(p1.loc[tk, PRICE_CLOSE])
        ed = pd.Timestamp(p0.loc[tk, PRICE_DATE])
        xd = pd.Timestamp(p1.loc[tk, PRICE_DATE])
        if xd > ed and ep > 0 and xp > 0:
            fwd[tk] = np.log(xp / ep)

    fwd_s  = pd.Series(fwd, name='fwd_ret')
    merged = pd.DataFrame({'pe': pe}).join(fwd_s, how='inner').dropna()
    if len(merged) < 20:
        continue

    # --- IC ---
    ic_val, _ = stats.spearmanr(merged['pe'], merged['fwd_ret'])
    ic_records.append({'date': T, 'ic': ic_val, 'n': len(merged)})

    # --- Quintile mean returns ---
    try:
        merged = merged.copy()
        merged['q'] = pd.qcut(
            merged['pe'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'], duplicates='drop'
        )
        qr = (
            merged.groupby('q', observed=True)['fwd_ret']
            .mean()
            .reindex(['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
        )
        qr.name = T
        qret_records.append(qr)
    except ValueError:
        pass

ic_df   = pd.DataFrame(ic_records).set_index('date')
ic_df.index = pd.DatetimeIndex(ic_df.index)

qret_df = pd.DataFrame(qret_records)
qret_df.index = pd.DatetimeIndex([r.name for r in qret_records])

print(f"Valid rebalance dates:  {len(ic_df)}")
print(f"Mean IC:               {ic_df['ic'].mean():.4f}")
print(f"IC std dev:            {ic_df['ic'].std():.4f}")
ic_tstat = ic_df['ic'].mean() / ic_df['ic'].std() * np.sqrt(len(ic_df))
print(f"IC t-stat:             {ic_tstat:.2f}  (|t| > 2 = statistically significant)")
print()
display(ic_df)

## Cell 9 — IC series chart

Each bar = Spearman IC at one rebalance date.  
Blue line = 4-period rolling average (smooths noise).  
Dashed = mean IC over the full period.

A consistently negative mean IC confirms P/E has predictive power in this sample.

In [ ]:
import plotly.graph_objects as go

mean_ic = ic_df['ic'].mean()
rolling = ic_df['ic'].rolling(4).mean()

fig = go.Figure()
fig.add_bar(
    x=ic_df.index, y=ic_df['ic'],
    name='IC',
    marker_color=['#e05c5c' if v < 0 else '#5ca0e0' for v in ic_df['ic']],
)
fig.add_scatter(
    x=rolling.index, y=rolling,
    name='4-period MA', line=dict(width=2, color='white'),
)
fig.add_hline(
    y=mean_ic, line_dash='dash', line_color='yellow',
    annotation_text=f'Mean IC = {mean_ic:.3f}',
    annotation_position='bottom right',
)
fig.update_layout(
    title='P/E — Spearman IC series',
    xaxis_title='Rebalance date',
    yaxis_title='IC (Spearman rank correlation)',
    template='plotly_dark',
    height=400,
)
fig.show()

## Cell 10 — Quintile cumulative return chart

At each date, we compute the **mean forward log return** within each quintile.
Cumulative sum = total compounded log return over the full backtest.

If value works:
- Q1 (cheapest) line should end highest
- Q5 (most expensive) line should end lowest
- A monotonic spread Q1 > Q2 > Q3 > Q4 > Q5 is the ideal pattern

In [ ]:
cumret = qret_df.fillna(0).cumsum()

colors = {'Q1': '#00c853', 'Q2': '#69f0ae', 'Q3': '#b0bec5', 'Q4': '#ff8a65', 'Q5': '#e53935'}

fig = go.Figure()
for q in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']:
    terminal = cumret[q].iloc[-1]
    fig.add_scatter(
        x=cumret.index, y=cumret[q],
        name=f'{q}  ({terminal:+.2f})',
        line=dict(width=2, color=colors[q]),
    )

fig.update_layout(
    title=f'P/E quintile cumulative log returns  (horizon = {HORIZON}d, {START} – {END})',
    xaxis_title='Rebalance date',
    yaxis_title='Cumulative log return',
    template='plotly_dark',
    height=450,
)
fig.show()

print("\nTerminal cumulative log return by quintile:")
display(cumret.iloc[[-1]].T.rename(columns={cumret.index[-1]: 'terminal'}).round(4))

## Summary: what each step did

| Cell | Step | Key decision |
|------|------|--------------|
| 3 | PIT alignment | Used `_eff` date (Publish Date or Report Date + 60d) to prevent lookahead bias |
| 4 | Cross-section | Dropped negative P/E — economically undefined for value sort |
| 5 | Quintiles | Equal-count bins; Q1 = cheapest, Q5 = most expensive |
| 6 | Forward return | Log(exit / entry), exit = nearest price ≤ T + 252d |
| 7 | IC | Spearman rank correlation between P/E and forward return at one date |
| 8 | Full loop | Same steps repeated at every rebalance date |
| 9 | IC series | Stability and sign of the signal over time |
| 10 | Quintile spread | Did cheap outperform expensive? Monotonic spread = signal works |